# Lecture 12: Common RAG Issues & Solutions

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Advanced  

---

## The Big Picture

In Lecture 11, you learned how to **measure** your RAG system with RAGAS.  
Now it's time to learn how to **fix** the problems you find.

> **Every RAG system has problems — the difference between a good engineer  
> and a great one is knowing how to fix them systematically.**

Think of yourself as a **car mechanic**.  
A customer says "my car isn't working." A bad mechanic randomly replaces  
parts hoping something works. A good mechanic runs diagnostics, finds the  
exact problem, and fixes just that one thing.

| Mechanic Approach | RAG Equivalent |
|-------------------|----------------|
| Listen to the engine | Read the RAGAS scores |
| Run diagnostics | Log retrieval + generation separately |
| Replace the bad part | Apply the targeted fix |
| Test drive | Re-run RAGAS to confirm |

### What You Will Learn

| # | Topic | What You'll Fix |
|---|-------|-----------------|
| 1 | The debugging mindset | How to think like a RAG mechanic |
| 2 | Poor retrieval quality | Wrong chunks coming back |
| 3 | Lost in the middle | LLM ignoring some retrieved docs |
| 4 | Hallucinations despite context | LLM making things up |
| 5 | Slow response times | Users waiting too long |
| 6 | Inconsistent results | Same question, different answers |
| 7 | Building a debugging toolkit | Your RAG maintenance kit |
| 8 | Hands-on: debug a broken RAG | Fix 3 real issues! |

> **After this lecture:** You'll be able to diagnose and fix the 5 most  
> common RAG problems using a systematic, data-driven approach.

---

## 0. Environment Setup

Run this cell **once** to install the packages we'll need today.

**Note:** This lecture requires an **OpenAI API key** (same as Lectures 10-11).  
We'll build several RAG variants to compare and fix issues.

In [1]:
# Install required packages (run once, then you can skip this cell)
%pip install langchain langchain-openai langchain-qdrant langchain-huggingface qdrant-client ragas datasets

  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/466.5 kB ? eta -:--:--
    --------------------------------------- 10.2/466.5 kB ? eta -:--:--
   --- ----------------------------------- 41.0/466.5 kB 495.5 kB/s eta 0:00:01
   --- ----------------------------------- 41.0/466.5 kB 495.5 kB/s eta 0:00:01
   ----- --------------------------------- 61.4/466.5 kB 328.2 kB/s eta 0:00:02
   ------ -------------------------------- 81.9/466.5 kB 383.3 kB/s eta 0:00:02
   --------- ---------------------------- 112.6/466.5 kB 437.6 kB/s eta 0:00:01
   --------- ---------------------------- 112.6/466.5 kB 437.6 kB/s eta 0:00:01
   ---------- --------------------------- 122.9/466.5 kB 328.4 kB/s eta 0:00:02
   ----------- -------------------------- 143.4/466.5 kB 341.3 kB/s eta 0:00:01
   -------------- 


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time

# Set your OpenAI API key (same as Lectures 10-11)
# Replace "your-api-key-here" with your actual key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# Verify the key is set
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key and api_key != "your-api-key-here":
    # [:8] shows only the first 8 characters for security
    print(f"API key set: {api_key[:20]}...")
else:
    print("WARNING: Please set your OpenAI API key above!")

---

## 1. The Debugging Mindset

Before we dive into specific issues, let's learn **how to think**  
about RAG problems. Random guessing wastes time. Follow this process:

### The 4-Step Debugging Process

```
Step 1: ISOLATE       Is it a retrieval problem or a generation problem?
           |                    |                       |
           v                    v                       v
Step 2: LOG          What docs were retrieved?   What prompt was sent?
           |                    |                       |
           v                    v                       v
Step 3: TEST         Create test cases from failing queries
           |                    |
           v                    v
Step 4: FIX          Change ONE variable at a time, then re-test
```

### The Golden Rule

| Approach | What You Do | Result |
|----------|-------------|--------|
| **Bad** (random) | Change chunk_size, k, prompt, model all at once | No idea what helped |
| **Good** (systematic) | Change ONLY chunk_size, re-test, then try next | Know exactly what works |

> **Rule: Change ONE variable at a time.**  
> If you change 3 things and the score improves,  
> you don't know which change actually helped.

### Quick Isolation Test

**How to tell if it's retrieval or generation:**

| Check | If Bad | Problem Is |
|-------|--------|------------|
| Look at retrieved chunks — are the right ones there? | Wrong chunks retrieved | **Retrieval** |
| Right chunks retrieved, but answer is wrong? | LLM misused context | **Generation** |
| RAGAS context_recall is low? | Missing relevant info | **Retrieval** |
| RAGAS faithfulness is low? | LLM is hallucinating | **Generation** |

Let's build a RAG system and practice this.

---

## 2. Build Our Base RAG System

We'll build a standard RAG system first, then deliberately break it  
in different ways to learn how to fix each problem.

In [3]:
# ============================================================
# BUILD THE BASE RAG SYSTEM (same as Lectures 10-11)
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# QDRANT CLOUD SETUP
# ============================================================
# Use the same credentials from previous lectures
# ============================================================

QDRANT_URL = ""
QDRANT_API_KEY = ""

# --- Load and chunk the knowledge base ---
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

# --- Create embeddings + vector store ---
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="nlp_course",
)

# --- Build a good RAG chain (our baseline) ---
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


good_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | good_prompt
    | llm
    | StrOutputParser()
)

print(f"Knowledge base: {len(chunks)} chunks in Qdrant Cloud")
print(f"Retriever: k=4, embedding=all-MiniLM-L6-v2")
print(f"LLM: gpt-4o-mini, temperature=0")
print(f"\nBase RAG system ready!")

C:\Users\Mani\AppData\Local\Temp\ipykernel_6620\2913260202.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
f:\AWFERA\.awfera\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 469.03it/s]
f:\AWFERA\.awfera\Lib\site-packages\qdrant_client\qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


ResponseHandlingException: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
# ============================================================
# BUILD BASE RAG SYSTEM — OpenAI Embeddings Version
# ============================================================
# EMBEDDING METHOD: OpenAI Embeddings (PAID, requires API key)
#
# This cell uses OpenAI embeddings (requires API key and costs money)
# For FREE local embeddings, use the previous cell instead
#
# Same baseline system, but with OpenAI embeddings
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# OPENAI API KEY (use the same key set in cell 3)
# ============================================================
# The key should already be set from cell 3
# ============================================================

# ============================================================
# QDRANT CLOUD SETUP
# ============================================================
QDRANT_URL = ""
QDRANT_API_KEY = ""

print("=" * 70)
print("BASE RAG SYSTEM - OpenAI Embeddings Version")
print("=" * 70)

# --- Load and chunk the knowledge base ---
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"[SUCCESS] Loaded and split: {len(chunks)} chunks")

# --- Create embeddings + vector store (OpenAI) ---
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")
print(f"[SUCCESS] OpenAI embeddings initialized (1536 dimensions)")

vectorstore_openai = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings_openai,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="nlp_course_openai",
)
print(f"[SUCCESS] Vector store created: nlp_course_openai")

# --- Build a good RAG chain (our baseline) ---
retriever_openai = vectorstore_openai.as_retriever(search_kwargs={"k": 4})


def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


good_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain_openai = (
    {"context": retriever_openai | format_docs, "question": RunnablePassthrough()}
    | good_prompt
    | llm
    | StrOutputParser()
)

print(f"[SUCCESS] RAG chain assembled")
print(f"\n" + "=" * 70)
print("BASE RAG SYSTEM READY - OpenAI Version")
print("=" * 70)
print(f"\nConfiguration:")
print(f"  Embeddings: OpenAI text-embedding-3-small (1536 dims)")
print(f"  Collection: nlp_course_openai")
print(f"  Retriever: k=4")
print(f"  LLM: gpt-4o-mini, temperature=0")
print(f"  Prompt: Strong grounding instructions")
print(f"\n[INFO] This is your baseline to compare against when debugging")
print(f"[INFO] Compare retrieval quality between free and paid embeddings!")

#### What just happened?

We built our **baseline RAG system** — the same one from Lectures 10-11:
- Loaded `nlp_article.txt` and split it into chunks (500 chars, 50 overlap)
- Stored chunks in Qdrant with HuggingFace embeddings
- Built the LCEL chain with `temperature=0` and a grounding prompt

This is our **"healthy car"** — we'll break things one at a time to learn  
how to diagnose and fix each problem.

---

## 3. The Debug Helper

Before we start breaking things, let's build a simple tool that  
shows us exactly what's happening inside the RAG chain.  
Think of this as our **mechanic's diagnostic scanner**.

In [ ]:
# ============================================================
# DEBUG HELPER — See inside your RAG chain
# ============================================================
# This function runs a question through RAG and shows you everything:
# - What chunks were retrieved (and their similarity scores)
# - What answer the LLM generated
# - How long each step took


def debug_rag(question, chain, ret, label="RAG"):
    """Run a question through RAG and show detailed debug info."""
    print(f"\n{'=' * 60}")
    print(f"DEBUG: {label}")
    print(f"{'=' * 60}")
    print(f"Question: {question}")

    # --- Step 1: Check what the retriever finds ---
    print(f"\n--- Retrieved Chunks ---")
    start = time.time()  # Start timer for retrieval
    docs = ret.invoke(question)
    retrieval_time = time.time() - start  # Calculate retrieval time

    # Show each retrieved chunk with its number and first 100 chars
    # enumerate() gives us index (i=0,1,2...) and the doc object
    for i, doc in enumerate(docs):
        # [:100] shows first 100 characters of the chunk
        print(f"  Chunk {i + 1}: {doc.page_content[:100]}...")

    print(f"  Retrieval time: {retrieval_time:.3f}s")

    # --- Step 2: Get the RAG answer ---
    print(f"\n--- RAG Answer ---")
    start = time.time()  # Start timer for generation
    answer = chain.invoke(question)
    total_time = time.time() - start  # Calculate total time

    print(f"  {answer}")
    print(f"  Total time: {total_time:.3f}s")
    print(f"{'=' * 60}")

    return answer, docs


# Test the debug helper with our base RAG
answer, docs = debug_rag(
    "What is the transformer architecture?",
    rag_chain,
    retriever,
    label="Base RAG",
)

#### What just happened?

We built a `debug_rag()` function that shows us **inside the RAG chain**:

| What It Shows | Why It Matters |
|---------------|----------------|
| Retrieved chunks | Are the right chunks being found? |
| Retrieval time | Is retrieval fast enough? |
| RAG answer | What did the LLM actually say? |
| Total time | Is the full pipeline fast enough? |

This is your **first debugging tool** — always start by looking at  
what chunks were retrieved before blaming the LLM.

> **Pro tip:** In production, log this info for every query.  
> When a user reports a bad answer, you can look up exactly what happened.

---

## 4. Issue 1: Poor Retrieval Quality

### The Problem

Your RAG answers "I don't have enough information" even though  
the answer IS in your documents. Or the answers are vague and incomplete.

### Symptoms

| Symptom | What's Happening |
|---------|------------------|
| "I don't have enough information" | Retriever can't find the right chunks |
| Vague, incomplete answers | Retrieved chunks don't contain full info |
| Low RAGAS context_recall | Missing relevant information |

### Common Causes

1. **Chunk size too large** — important details buried in big chunks
2. **Chunk size too small** — context split across multiple tiny chunks
3. **k too low** — not retrieving enough chunks
4. **Bad embedding model** — poor semantic matching

Let's test how chunk size affects retrieval!

In [ ]:
# ============================================================
# EXPERIMENT: How does chunk_size affect retrieval?
# ============================================================
# We'll build 3 RAG systems with different chunk sizes
# and compare what chunks they retrieve for the same question.

test_question = "What is Named Entity Recognition?"
print(f"Question: {test_question}")
print(f"(The answer is in our article — let's see what each retriever finds)\n")

# Test 3 different chunk sizes
# range() would work but a list is clearer when values aren't sequential
chunk_sizes = [200, 500, 1000]

for size in chunk_sizes:
    print(f"{'=' * 60}")
    print(f"Chunk Size: {size} characters")
    print(f"{'=' * 60}")
    # Build a new splitter with this chunk size
    test_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=50,
    )
    test_chunks = test_splitter.split_documents(documents)
    print(f"  Total chunks created: {len(test_chunks)}")

    # Build a new vector store with these chunks
    test_vectorstore = QdrantVectorStore.from_documents(
        documents=test_chunks,
        embedding=embeddings,
        location=":memory:",
        collection_name=f"test_{size}",
    )

    # Retrieve top 3 chunks
    test_retriever = test_vectorstore.as_retriever(search_kwargs={"k": 3})
    results = test_retriever.invoke(test_question)

    # Show what was retrieved
    # enumerate() gives us the index and the doc object
    for i, doc in enumerate(results):
        # len() shows how long the chunk is
        print(f"  Chunk {i + 1} ({len(doc.page_content)} chars):")
        # [:120] shows first 120 characters
        print(f"    {doc.page_content[:120]}...")
    print()

#### What just happened?

We tested **3 different chunk sizes** with the same question:

| Chunk Size | Effect |
|------------|--------|
| **200 chars** (too small) | More chunks, but each is a tiny fragment — context is split |
| **500 chars** (good balance) | Each chunk has enough context to be meaningful |
| **1000 chars** (large) | Fewer chunks, but may include unrelated info in same chunk |

### The Fix

| Setting | Recommendation |
|---------|----------------|
| `chunk_size` | Start with 500, test 300-1000 range |
| `chunk_overlap` | Use 10-20% of chunk_size (e.g., 50-100 for size=500) |
| `k` | Start with 4, increase to 7-10 if recall is low |

> **Key lesson:** There's no magic chunk size. Test different values  
> and measure with RAGAS to find the best one for YOUR data.

---

## 6. Issue 3: Hallucinations Despite Context

### The Problem

The retriever finds the right chunks, but the LLM **makes up facts**  
that aren't in the context. This is the most dangerous RAG failure  
because the answer looks confident but is wrong.

### Symptoms

| Symptom | RAGAS Metric |
|---------|-------------|
| Answer contains facts not in retrieved chunks | Low faithfulness |
| Answer mixes real info with made-up details | Low faithfulness |
| Answer says specific dates/numbers not in context | Low faithfulness |

### The #1 Cause: Weak Prompts

Compare these two prompts:

| Prompt | Risk |
|--------|------|
| "Answer the question using the context" | High — LLM may add its own knowledge |
| "Answer ONLY from the context. If not found, say 'I don't know'" | Low — LLM is constrained |

Let's test this!

In [ ]:
# ============================================================
# DEMO: Weak Prompt vs Strong Prompt
# ============================================================
# A weak prompt lets the LLM add its own knowledge (hallucinate).
# A strong prompt forces it to stick to the context.

# --- Weak prompt (invites hallucination) ---
weak_prompt = ChatPromptTemplate.from_template(
    """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""
)

# --- Strong prompt (prevents hallucination) ---
strong_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following context.
Do NOT add any information that is not explicitly stated in the context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
)

# Build two chains — same retriever, different prompts
chain_weak = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | weak_prompt
    | llm
    | StrOutputParser()
)

chain_strong = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | strong_prompt
    | llm
    | StrOutputParser()
)

# Ask a question where the LLM might add extra details
# GPT-3 info IS in our article, but the LLM might add extra facts
test_question = "what is NLP?"
# test_question = "who is Elon musk ?"

print("COMPARISON: Weak Prompt vs Strong Prompt")
print(f"Question: {test_question}\n")

answer_weak = chain_weak.invoke(test_question)
print(f"--- Weak Prompt Answer ---")
print(f"{answer_weak}\n")

answer_strong = chain_strong.invoke(test_question)
print(f"--- Strong Prompt Answer ---")
print(f"{answer_strong}\n")

print("Notice: The weak prompt may include facts NOT in our article.")
print("The strong prompt sticks to what's actually in the context.")

#### What just happened?

We tested the same question with two different prompts:

| Prompt | Behavior |
|--------|----------|
| **Weak** | LLM may add facts from its training data (hallucination) |
| **Strong** | LLM stays grounded in the retrieved context |

### The Anti-Hallucination Checklist

| Fix | What to Do |
|-----|------------|
| **Prompt** | Add "ONLY use the context" and "say I don't know if not found" |
| **Temperature** | Set `temperature=0` (no randomness for factual answers) |
| **System message** | Reinforce grounding in the system prompt |
| **Model choice** | Some models are more faithful than others |

> **Faithfulness is the #1 metric to get right.**  
> A RAG system that makes things up is worse than no RAG system at all.

In [ ]:
# ============================================================
# DEMO: Temperature 0 vs Temperature 0.7
# ============================================================
# temperature=0 means deterministic (always picks the most likely word)
# temperature=0.7 means more creative (picks less likely words sometimes)

# For factual RAG, temperature=0 is almost always better
llm_temp0 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_temp07 = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

chain_temp0 = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | good_prompt
    | llm_temp0
    | StrOutputParser()
)

chain_temp07 = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | good_prompt
    | llm_temp07
    | StrOutputParser()
)

test_question = "what is NLP?  "

print("COMPARISON: Temperature 0 vs Temperature 0.7")
print(f"Question: {test_question}\n")

# Run each 3 times to show consistency vs variability
# range(3) generates 0, 1, 2 for three runs
print("--- Temperature 0 (3 runs) ---")
for run in range(3):
    answer = chain_temp0.invoke(test_question)
    # [:100] shows first 100 characters
    print(f"  Run {run + 1}: {answer[:100]}")

print("\n--- Temperature 0.7 (3 runs) ---")
for run in range(3):
    answer = chain_temp07.invoke(test_question)
    print(f"  Run {run + 1}: {answer[:100]}")

print("\nNotice: temp=0 gives the SAME answer every time.")
print("temp=0.7 may give slightly different answers each run.")

#### What just happened?

We ran the same question **3 times** with each temperature setting:

| Temperature | Behavior | Use For |
|-------------|----------|---------|
| **0** | Same answer every time (deterministic) | Factual RAG, production systems |
| **0.7** | Slightly different each time (creative) | Creative writing, brainstorming |

> **For factual RAG, always use `temperature=0`.**  
> Users expect the same question to give the same answer.

---

## 7. Issue 4: Slow Response Times

### The Problem

Users are waiting 5-10 seconds for answers. In production,  
anything over 3 seconds feels slow.

### Common Causes

| Cause | Why It's Slow |
|-------|---------------|
| Large k value | More chunks = bigger prompt = slower LLM |
| No streaming | User stares at a blank screen |
| Re-computing embeddings | Same query computed again and again |

### Fix 1: Use Streaming

Streaming doesn't make the total time faster, but it makes the  
**user experience** much better. Users see words appearing immediately  
instead of waiting for the complete answer.

In [ ]:
# ============================================================
# DEMO: Regular vs Streaming Response
# ============================================================
# Streaming shows words as they're generated — much better UX!

test_question = "What is NLP?"

# --- Regular (wait for full answer) ---
print("--- Regular (wait for full answer) ---")
start = time.time()
answer = rag_chain.invoke(test_question)
regular_time = time.time() - start
print(f"{answer}")
print(f"Time: {regular_time:.2f}s (user waited this long to see ANYTHING)\n")

# --- Streaming (see words as they appear) ---
print("--- Streaming (words appear as generated) ---")
start = time.time()

# .stream() returns chunks of text as they're generated
# end="" prevents adding newline after each chunk
# flush=True forces Python to display each chunk immediately
for chunk in rag_chain.stream(test_question):
    print(chunk, end="", flush=True)

stream_time = time.time() - start
print(f"\nTime: {stream_time:.2f}s (but user saw first word in <1s!)\n")

print(f"Total time is similar, but streaming FEELS much faster.")
print(f"Users see progress immediately instead of staring at a blank screen.")

#### What just happened?

We compared two ways to show the answer:

| Method | User Experience |
|--------|----------------|
| **Regular** | Blank screen for 3-5 seconds, then full answer appears |
| **Streaming** | Words start appearing in less than 1 second |

### Other Speed Fixes

| Fix | How It Helps | Example |
|-----|-------------|--------|
| **Lower k** | Less context = faster LLM response | k=4 instead of k=10 |
| **Cache queries** | Same question returns instant cached answer | Store results in a dict |
| **Smaller model** | Faster generation, slightly lower quality | gpt-4o-mini instead of gpt-4o |

> **Always use streaming in user-facing applications.**  
> It's the single biggest UX improvement you can make.

---

## 8. Issue 5: Inconsistent Results

### The Problem

A user asks the same question twice and gets **different answers**.  
This destroys user trust.

### Causes & Fixes

| Cause | Fix |
|-------|-----|
| High temperature | Set `temperature=0` |
| Using "latest" model version | Pin exact model version |
| Borderline similarity scores | Add a minimum score threshold |
| Different question phrasing | Normalize questions before querying |

We already saw temperature in the hallucination section.  
Let's look at **question normalization** — a simple but powerful trick.

In [ ]:
# ============================================================
# DEMO: Question Normalization
# ============================================================
# Users ask the same thing in different ways.
# Normalizing questions helps get consistent results.


def normalize_question(question):
    """Clean up a question for more consistent retrieval."""
    # Step 1: Convert to lowercase
    question = question.lower()

    # Step 2: Remove extra whitespace
    # split() breaks on any whitespace, join() puts back with single spaces
    question = " ".join(question.split())

    # Step 3: Strip leading/trailing whitespace
    question = question.strip()

    return question


# These are all the same question, written differently
variations = [
    "What is BERT?",
    "  what is BERT?  ",      # Extra whitespace
    "WHAT IS BERT?",          # ALL CAPS
    "what   is   bert?",      # Multiple spaces
]

print("Question Normalization Demo")
print(f"{'=' * 60}\n")

# Show how each variation gets normalized to the same string
for original in variations:
    normalized = normalize_question(original)
    # repr() shows the string with quotes, making whitespace visible
    print(f"  Original:   {repr(original)}")
    print(f"  Normalized: {repr(normalized)}")
    # Check if normalized version matches the first one
    match = "(same)" if normalized == normalize_question(variations[0]) else "(different!)"
    print(f"  Match: {match}\n")

print("All variations normalize to the same string.")
print("This means they'll get the same cache hit and same retrieval results.")

Question Normalization Demo

  Original:   'What is BERT?'
  Normalized: 'what is bert?'
  Match: (same)

  Original:   '  what is BERT?  '
  Normalized: 'what is bert?'
  Match: (same)

  Original:   'WHAT IS BERT?'
  Normalized: 'what is bert?'
  Match: (same)

  Original:   'what   is   bert?'
  Normalized: 'what is bert?'
  Match: (same)

All variations normalize to the same string.
This means they'll get the same cache hit and same retrieval results.


#### What just happened?

We built a **question normalizer** that cleans up user input:

| Step | What It Does | Example |
|------|-------------|--------|
| Lowercase | Removes case differences | "BERT" → "bert" |
| Strip whitespace | Removes extra spaces | "  what  is  " → "what is" |
| Normalize spaces | Single spaces only | "what   is" → "what is" |

Combined with caching, this means:
- "What is BERT?" → gets answer, saves to cache
- "WHAT IS BERT?" → normalized → cache hit → same answer instantly

> **Consistency builds user trust.** Small preprocessing steps  
> like normalization make a big difference in production.

---

## 9. Building Your Debugging Toolkit

Every RAG developer needs these tools ready. Think of this as  
your **mechanic's toolbox** — always keep it nearby.

### Essential Logging (Log These for Every Query)

| What to Log | Why |
|-------------|-----|
| Retrieved document IDs + scores | Know what the retriever found |
| Full prompt sent to LLM | See exactly what the LLM received |
| LLM response + latency | Track quality and speed |
| User question (normalized) | Track common queries |

### The Test Suite Approach

```
1. Create 20+ test questions with expected answers
2. Run RAGAS after every change
3. Track metric improvements over time
4. Document what worked and what didn't
```

### The Change Log

| Date | Change Made | Metric Before | Metric After | Keep? |
|------|------------|--------------|-------------|-------|
| Day 1 | chunk_size 500 → 300 | recall: 0.65 | recall: 0.72 | Yes |
| Day 2 | k=4 → k=8 | precision: 0.80 | precision: 0.60 | No (revert) |
| Day 3 | Added stronger prompt | faith: 0.75 | faith: 0.90 | Yes |

> **Document everything.** Your future self will thank you  
> when you need to remember why you made a change.

In [ ]:
# ============================================================
# BUILD: A Complete RAG Logger
# ============================================================
# This logs everything you need to debug RAG issues.
# In production, you'd save these logs to a file or database.
# Includes retry logic to handle network timeouts gracefully.


def ask_and_log(question, chain, ret, log_list, max_retries=3):
    """Run a RAG query and log everything for debugging.

    Includes retry logic to handle network timeouts.
    """
    # Normalize the question for consistency
    normalized_q = normalize_question(question)

    # Retry logic for network timeouts
    for attempt in range(1, max_retries + 1):
        try:
            # Time the retrieval step
            start = time.time()
            docs = ret.invoke(normalized_q)
            retrieval_time = time.time() - start

            # Time the full chain
            start = time.time()
            answer = chain.invoke(question)
            total_time = time.time() - start

            # Build the log entry (a dictionary with all the info)
            log_entry = {
                "question": question,
                "normalized": normalized_q,
                "num_chunks": len(docs),
                # Store first 80 chars of each retrieved chunk
                "chunks_preview": [doc.page_content[:80] for doc in docs],
                "answer": answer,
                "retrieval_time": round(retrieval_time, 3),
                "total_time": round(total_time, 3),
            }

            # Add to our log list
            log_list.append(log_entry)

            return answer

        except Exception as e:
            if attempt < max_retries:
                wait_time = attempt * 5  # Wait 5s, 10s, 15s
                print(f"  [RETRY] Attempt {attempt} failed: {type(e).__name__}")
                print(f"  [RETRY] Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"  [ERROR] All {max_retries} attempts failed: {type(e).__name__}")
                print(f"  [ERROR] {str(e)[:100]}")
                return f"[ERROR] Query failed after {max_retries} retries"


# Create a log list and run some questions
query_log = []

test_questions = [
    "What is NLP?",
    "How does BERT work?",
    "What is RAG?",
]

print("Running 3 questions with full logging...\n")

# Run each question and log everything
# enumerate() gives us index (i=0,1,2) and the question string
for i, q in enumerate(test_questions):
    answer = ask_and_log(q, rag_chain, retriever, query_log)
    # [:60] shows first 60 characters of the answer
    print(f"  Q{i + 1}: {q}")
    print(f"  A: {answer[:60]}...\n")

# Show the logged data
print(f"{'=' * 60}")
print(f"QUERY LOG SUMMARY ({len(query_log)} queries)")
print(f"{'=' * 60}")

# Display log entries in a readable format
for entry in query_log:
    print(f"\n  Question: {entry['question']}")
    print(f"  Chunks retrieved: {entry['num_chunks']}")
    print(f"  Retrieval time: {entry['retrieval_time']}s")
    print(f"  Total time: {entry['total_time']}s")


Running 3 questions with full logging...

  Q1: What is NLP?
  A: NLP, or Natural Language Processing, combines computational ...

  Q2: How does BERT work?
  A: BERT works by reading text bidirectionally, meaning it consi...

  Q3: What is RAG?
  A: Retrieval-Augmented Generation (RAG) combines retrieval and ...

QUERY LOG SUMMARY (3 queries)

  Question: What is NLP?
  Chunks retrieved: 4
  Retrieval time: 0.977s
  Total time: 2.168s

  Question: How does BERT work?
  Chunks retrieved: 4
  Retrieval time: 0.319s
  Total time: 1.964s

  Question: What is RAG?
  Chunks retrieved: 4
  Retrieval time: 0.313s
  Total time: 1.934s


#### What just happened?

We built a **complete RAG logger** that records:

| Logged Item | Why It's Useful |
|-------------|----------------|
| Original + normalized question | Debugging cache and retrieval |
| Number of chunks retrieved | Checking if k is appropriate |
| Chunk previews | Quickly see if right content was found |
| Answer text | Review quality |
| Timing info | Identify slow queries |

In production, save these logs to a file or database.  
When a user reports a bad answer, you can look up exactly what happened.

> **Logging is not optional in production RAG.**  
> Without logs, you're debugging blind.

---

## 10. Hands-On: Debug a Broken RAG System

Now let's put your skills to the test!

We'll create a RAG system with **3 intentional problems**.  
Your job: identify each problem and apply the correct fix.

### The 3 Hidden Issues

| Issue # | Hint |
|---------|------|
| 1 | Look at the chunk size |
| 2 | Look at the prompt |
| 3 | Look at the temperature |

In [ ]:
# ============================================================
# THE BROKEN RAG SYSTEM — Can you spot the 3 issues?
# ============================================================

# Issue 1: Something wrong with chunking...
bad_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,       # Suspiciously small!
    chunk_overlap=0,     # No overlap!
)
bad_chunks = bad_splitter.split_documents(documents)

bad_vectorstore = QdrantVectorStore.from_documents(
    documents=bad_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="broken_rag",
)

bad_retriever = bad_vectorstore.as_retriever(search_kwargs={"k": 3})

# Issue 2: Something wrong with the prompt...
bad_prompt = ChatPromptTemplate.from_template(
    """Answer this question: {question}

Here is some context: {context}"""
)

# Issue 3: Something wrong with the LLM settings...
bad_llm = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)

# Build the broken chain
broken_chain = (
    {"context": bad_retriever | format_docs, "question": RunnablePassthrough()}
    | bad_prompt
    | bad_llm
    | StrOutputParser()
)

print(f"Broken RAG system built!")
print(f"Chunks created: {len(bad_chunks)} (compare to {len(chunks)} in the good system)")
print(f"\nLet's test it...\n")

# Test with a question
test_q = "What is Named Entity Recognition and give an example?"

print(f"Question: {test_q}\n")

# Show what the broken retriever finds
broken_docs = bad_retriever.invoke(test_q)
print("--- Retrieved Chunks (broken) ---")
for i, doc in enumerate(broken_docs):
    # These chunks will be tiny fragments!
    print(f"  Chunk {i + 1} ({len(doc.page_content)} chars): {doc.page_content}")

# Get the answer
broken_answer = broken_chain.invoke(test_q)
print(f"\n--- Broken RAG Answer ---")
print(f"{broken_answer}")

# Compare with the good system
good_answer = rag_chain.invoke(test_q)
print(f"\n--- Good RAG Answer ---")
print(f"{good_answer}")

Broken RAG system built!
Chunks created: 180 (compare to 21 in the good system)

Let's test it...

Question: What is Named Entity Recognition and give an example?

--- Retrieved Chunks (broken) ---
  Chunk 1 (45 chars): identifying and classifying named entities in
  Chunk 2 (45 chars): Named Entity Recognition (NER) is the task of
  Chunk 3 (46 chars): Natural Language Processing, commonly known as

--- Broken RAG Answer ---
Named Entity Recognition (NER) is a task in Natural Language Processing (NLP) that involves identifying and classifying named entities in text into predefined categories such as people, organizations, locations, dates, and other specific items. The goal of NER is to extract meaningful information from unstructured text, enabling better understanding and data manipulation.

**Example:**  
Consider the following sentence:

"Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in April 1976 in Cupertino, California."

In this example, a NER system wo

#### What just happened?

We built a RAG system with **3 intentional problems**:

| Issue | What's Wrong | The Fix |
|-------|-------------|--------|
| **1. Tiny chunks** | `chunk_size=50` creates fragments, not meaningful text | Use `chunk_size=500` with `chunk_overlap=50` |
| **2. Weak prompt** | No instruction to stay grounded in context | Add "ONLY use the context" + "say I don't know" |
| **3. High temperature** | `temperature=1.0` adds randomness and hallucination | Use `temperature=0` for factual RAG |

The broken system's chunks were so small they lost all meaning.  
Combined with a weak prompt and high temperature, the LLM was  
essentially guessing instead of using the retrieved context.

In [ ]:
# ============================================================
# THE FIX — Apply all 3 corrections
# ============================================================

# Fix 1: Proper chunk size
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Good size for meaningful chunks
    chunk_overlap=50,     # Overlap to avoid splitting key info
)
fixed_chunks = fixed_splitter.split_documents(documents)

fixed_vectorstore = QdrantVectorStore.from_documents(
    documents=fixed_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="fixed_rag",
)

fixed_retriever = fixed_vectorstore.as_retriever(search_kwargs={"k": 4})

# Fix 2: Strong grounding prompt
fixed_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following context.
Do NOT add any information that is not explicitly stated in the context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
)

# Fix 3: temperature=0 for factual answers
fixed_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Build the fixed chain
fixed_chain = (
    {"context": fixed_retriever | format_docs, "question": RunnablePassthrough()}
    | fixed_prompt
    | fixed_llm
    | StrOutputParser()
)

print(f"Fixed RAG system built!")
print(f"Chunks: {len(bad_chunks)} (broken) -> {len(fixed_chunks)} (fixed)")
print(f"Prompt: weak -> strong grounding")
print(f"Temperature: 1.0 -> 0\n")

# Test with the same question
test_q = "What is Named Entity Recognition and give an example?"
print(f"Question: {test_q}\n")

# Show fixed retrieval
fixed_docs = fixed_retriever.invoke(test_q)
print("--- Retrieved Chunks (fixed) ---")
for i, doc in enumerate(fixed_docs):
    # [:100] shows first 100 characters of each chunk
    print(f"  Chunk {i + 1} ({len(doc.page_content)} chars): {doc.page_content[:100]}...")

# Get the fixed answer
fixed_answer = fixed_chain.invoke(test_q)
print(f"\n--- Fixed RAG Answer ---")
print(f"{fixed_answer}")

Fixed RAG system built!
Chunks: 180 (broken) -> 21 (fixed)
Prompt: weak -> strong grounding
Temperature: 1.0 -> 0

Question: What is Named Entity Recognition and give an example?

--- Retrieved Chunks (fixed) ---
  Chunk 1 (449 chars): Named Entity Recognition (NER) is the task of identifying and classifying named entities in text int...
  Chunk 2 (427 chars): Chapter 2: Core NLP Tasks

Text Classification is one of the most fundamental tasks in NLP. It invol...
  Chunk 3 (425 chars): Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?
...
  Chunk 4 (346 chars): NLP combines computational linguistics — rule-based modeling of human language — with statistical, m...

--- Fixed RAG Answer ---
Named Entity Recognition (NER) is the task of identifying and classifying named entities in text into predefined categories such as person names, organizations, locations, dates, and monetary values. For example, in the sentence "Apple Inc. was founded 

#### What just happened?

We applied all 3 fixes and the RAG system is working properly again:

| Setting | Broken | Fixed |
|---------|--------|-------|
| chunk_size | 50 (tiny fragments) | 500 (meaningful chunks) |
| chunk_overlap | 0 (info gets split) | 50 (smooth transitions) |
| Prompt | Weak (no grounding) | Strong ("ONLY use context") |
| Temperature | 1.0 (random) | 0 (deterministic) |

The fixed answer should be accurate, grounded in the context,  
and consistent across multiple runs.

In [ ]:
# ============================================================
# EVALUATE: Compare broken vs fixed with RAGAS
# ============================================================

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from datasets import Dataset

# RAGAS requires explicit LLM and embeddings for evaluation
ragas_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
ragas_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Small test set for quick comparison (3 questions)
eval_questions = [
    "What is Natural Language Processing?",
    "What is Named Entity Recognition?",
    "When was the transformer architecture introduced?",
]

eval_ground_truths = [
    "NLP is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language.",
    "Named Entity Recognition (NER) is the task of identifying and classifying named entities in text into categories such as person names, organizations, locations, dates, and monetary values.",
    "The transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017.",
]


def evaluate_chain(chain, ret, label, max_retries=3):
    """Run RAGAS evaluation on a chain and return the results."""
    answers = []
    contexts = []

    # Run each question through the chain with retry logic
    for q in eval_questions:
        for attempt in range(1, max_retries + 1):
            try:
                answers.append(chain.invoke(q))
                docs = ret.invoke(q)
                contexts.append([doc.page_content for doc in docs])
                break
            except Exception as e:
                if attempt < max_retries:
                    print(f"  [RETRY] {q[:30]}... attempt {attempt} failed, waiting {attempt * 5}s...")
                    time.sleep(attempt * 5)
                else:
                    print(f"  [ERROR] {q[:30]}... failed after {max_retries} attempts")
                    answers.append("Error: query failed")
                    contexts.append(["No context retrieved"])

    # Create RAGAS dataset
    eval_data = Dataset.from_dict({
        "question": eval_questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": eval_ground_truths,
    })

    # Run evaluation with explicit LLM and embeddings
    result = evaluate(
        dataset=eval_data,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )

    # Use .to_pandas() since EvaluationResult doesn't support .items()
    df_results = result.to_pandas()

    print(f"\n--- {label} ---")
    metric_scores = {}
    for metric_name in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
        if metric_name in df_results.columns:
            score = df_results[metric_name].mean()
            metric_scores[metric_name] = score
            print(f"  {metric_name:<25} {score:.4f}")

    return metric_scores


print("Evaluating both systems with RAGAS...")
print("(This takes 2-4 minutes for both evaluations)")

# Evaluate the broken system
broken_result = evaluate_chain(broken_chain, bad_retriever, "BROKEN RAG")

# Evaluate the fixed system
fixed_result = evaluate_chain(fixed_chain, fixed_retriever, "FIXED RAG")

# Show the comparison
print(f"\n{'=' * 60}")
print(f"BEFORE vs AFTER COMPARISON")
print(f"{'=' * 60}")
print(f"{'Metric':<25} {'Broken':<10} {'Fixed':<10} {'Change':<10}")
print(f"{'-' * 55}")

# Compare each metric
for metric in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    if metric in broken_result and metric in fixed_result:
        broken_score = broken_result[metric]
        fixed_score = fixed_result[metric]
        change = fixed_score - broken_score
        direction = "+" if change > 0 else ""
        print(f"  {metric:<25} {broken_score:<10.4f} {fixed_score:<10.4f} {direction}{change:.4f}")


C:\Users\mhari\AppData\Local\Temp\ipykernel_12320\1943888651.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\mhari\AppData\Local\Temp\ipykernel_12320\1943888651.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\mhari\AppData\Local\Temp\ipykernel_12320\1943888651.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from

Evaluating both systems with RAGAS...
(This takes 2-4 minutes for both evaluations)


Evaluating:   8%|▊         | 1/12 [00:07<01:17,  7.09s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 12/12 [00:38<00:00,  3.23s/it]



--- BROKEN RAG ---
  faithfulness              0.0655
  answer_relevancy          0.8831
  context_precision         0.0000
  context_recall            0.3333


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 12/12 [00:32<00:00,  2.72s/it]



--- FIXED RAG ---
  faithfulness              1.0000
  answer_relevancy          0.9225
  context_precision         1.0000
  context_recall            1.0000

BEFORE vs AFTER COMPARISON
Metric                    Broken     Fixed      Change    
-------------------------------------------------------
  faithfulness              0.0655     1.0000     +0.9345
  answer_relevancy          0.8831     0.9225     +0.0394
  context_precision         0.0000     1.0000     +1.0000
  context_recall            0.3333     1.0000     +0.6667


#### What just happened?

We ran RAGAS on both the **broken** and **fixed** RAG systems:

The fixed system should show **significant improvements** across all metrics.

This is the **data-driven optimization cycle** in action:

```
1. Identify problems (RAGAS scores)
2. Apply targeted fixes
3. Measure improvement (RAGAS again)
4. Repeat
```

> **This is exactly what you'll do in real projects.**  
> Build → Evaluate → Fix → Re-evaluate → Ship.

---

## 11. Mini Challenges

### Challenge 1: Find the Best Chunk Size
Test chunk sizes of 300, 500, 700, and 1000. Use the 3 evaluation  
questions above and run RAGAS for each. Which chunk size gets the  
highest overall scores?

### Challenge 2: Build a Better Logger
Extend the `ask_and_log()` function to also log:
- The full prompt sent to the LLM (context + question)
- Whether the answer contains "I don't have enough information"
- A warning if total_time > 3 seconds

### Challenge 3: The Prompt Engineering Challenge
Write 3 different prompts and test them with RAGAS.  
Which prompt gives the highest faithfulness score?  
Try: different system roles, different grounding instructions,  
adding "think step by step" — measure everything!

> **Hint for Challenge 3:** The best prompts are usually short,  
> specific, and include clear constraints like "ONLY use the context".

In [ ]:
# ============================================================
# Challenge 1: Find the Best Chunk Size
# ============================================================
# Test chunk sizes: 300, 500, 700, 1000
# Run RAGAS for each and compare scores

# Your code here:


In [ ]:
# ============================================================
# Challenge 2: Build a Better Logger
# ============================================================
# Extend ask_and_log() with:
# - Full prompt logging
# - "I don't know" detection
# - Slow query warning

# Your code here:


In [ ]:
# ============================================================
# Challenge 3: Prompt Engineering Challenge
# ============================================================
# Write 3 different prompts, test each with RAGAS
# Which one has the highest faithfulness?

# Your code here:


---

## 12. Quick Reference: RAG Debugging Cheat Sheet

### The 5 Most Common Issues

| # | Issue | Symptom | Metric | Fix |
|---|-------|---------|--------|-----|
| 1 | Poor retrieval | "I don't know" answers | Low context_recall | Adjust chunk_size, increase k |
| 2 | Lost in middle | LLM ignores some docs | Low faithfulness | Reduce k, reorder chunks |
| 3 | Hallucinations | Made-up facts | Low faithfulness | Strong prompt, temp=0 |
| 4 | Slow responses | Users waiting >3s | N/A (timing) | Streaming, caching, lower k |
| 5 | Inconsistent | Different answers | Varies | temp=0, normalize questions |

### The Debugging Process

```python
# Step 1: Isolate the problem
docs = retriever.invoke(question)   # Check retrieval
answer = chain.invoke(question)     # Check generation

# Step 2: Log everything
print(f"Chunks: {len(docs)}")
for doc in docs:
    print(doc.page_content[:100])

# Step 3: Run RAGAS
result = evaluate(dataset=dataset, metrics=[faithfulness, ...])

# Step 4: Apply ONE fix, re-test
# Change chunk_size OR k OR prompt — not all at once!
```

### Production Checklist

```
[ ] temperature=0 for factual RAG
[ ] Strong grounding prompt with "ONLY use context"
[ ] chunk_size tested (300-1000 range)
[ ] k value tested (3-7 range)
[ ] Streaming enabled for user-facing apps
[ ] Query logging enabled
[ ] Test suite of 20+ questions with RAGAS
[ ] Cache for repeated queries
[ ] Question normalization
```

---

## 13. Key Takeaways

1. **Systematic debugging:** Always isolate (retrieval vs generation), log, test, then fix
2. **Change ONE variable at a time** — otherwise you won't know what helped
3. **Most failures have known causes and known fixes** — use the cheat sheet
4. **Logging + evaluation are essential** — you can't fix what you can't see
5. **Streaming is the biggest UX win** — always use it in user-facing apps
6. **Caching + normalization** save time and money in production

### The Complete RAG Optimization Flow

```
Build RAG (L10) --> Evaluate with RAGAS (L11) --> Debug & Fix (L12)
      ^                                                  |
      |__________________________________________________|     
              (measure, fix, re-measure, repeat)
```

### Next Lecture

**Lecture 13: Hybrid Search** — Combine dense vector search with  
keyword search for even better retrieval. The best of both worlds!

---

*Hope to Skill — Building the future, one skill at a time.*

---

## Appendix: PEP 8 Style Rules Used in This Notebook

All code in this notebook follows Python's PEP 8 style guide.  
Here are the rules applied, with examples from the code above.

### Naming Conventions

| Rule | Convention | Example from This Notebook |
|------|-----------|---------------------------|
| Variables & functions | `snake_case` | `debug_rag()`, `query_log`, `normalize_question()` |
| Constants | `UPPER_CASE` | `OPENAI_API_KEY` (environment variable) |
| Classes | `PascalCase` | `ChatOpenAI`, `Dataset`, `StrOutputParser` (from libraries) |

### Import Rules

| Rule ID | Rule | Example |
|---------|------|---------|
| E401 | One import per line | `from ragas import evaluate` |
| E402 | Imports at the top of each section | All imports at cell top |
| — | Group: stdlib then third-party then local | `os`, `time` then `ragas` then `langchain` |

### Whitespace & Formatting

| Rule ID | Rule | Example |
|---------|------|---------|
| E225 | Spaces around operators | `score >= 0.8`, `i + 1` |
| E231 | Space after commas | `evaluate(dataset=eval_data, metrics=[...])` |
| E265 | Block comments start with `# ` | `# Build the broken chain` |
| E303 | Two blank lines before function defs | See `debug_rag()` definition |
| E501 | Max line length of 79 characters | Long strings use wrapping |

### Best Practices Used

| Practice | Why | Example |
|----------|-----|---------|
| f-strings | Clean string formatting | `f"Score: {score:.4f}"` |
| `enumerate()` | Index + value in loops | `for i, doc in enumerate(docs)` |
| List comprehensions | Concise list building | `[doc.page_content[:80] for doc in docs]` |
| `:<25` formatting | Align columns in output | `f"{metric_name:<25} {score:.4f}"` |
| Docstrings | Explain function purpose | `debug_rag()` has a docstring |
| Descriptive names | Code reads like English | `query_log` not `ql`, `fixed_chain` not `fc` |
| `repr()` | Show strings with quotes | `repr(original)` to make whitespace visible |
| Dictionary for config | Map names to actions | `fixes = {"faithfulness": [...]}` |
| `time.time()` for timing | Measure performance | `start = time.time()` |

### Quick PEP 8 Cheat Sheet

```python
# GOOD (PEP 8 compliant)
query_log = []
for i, question in enumerate(test_questions):
    answer = ask_and_log(question, rag_chain, retriever, query_log)
    print(f"  Q{i + 1}: {answer[:60]}...")

# BAD (violates PEP 8)
ql = []
for i,q in enumerate(test_questions):           # no space after comma
    a = ask_and_log(q,rag_chain,retriever,ql)   # unclear variable names
    print("  Q"+str(i+1)+": "+a[:60]+"...")    # string concat, no f-string
```